In [4]:
# redis_demo.py
import json
from datetime import datetime

import redis

# ---------- 配置 ----------
REDIS_HOST = "127.0.0.1"
REDIS_PORT = 6379
REDIS_PASSWORD = None   # 有密码就填
REDIS_DB = 0

# ---------- 连接 ----------
r = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD,
    db=REDIS_DB,
    decode_responses=True,   # 返回 str 而不是 bytes
)


def add_message(session_id: str, user: str, message: str):
    """追加一条对话到 Redis List"""
    key = f"chat:history:{session_id}"
    record = {
        "timestamp": datetime.utcnow().isoformat(),
        "user": user,
        "message": message,
    }
    r.rpush(key, json.dumps(record, ensure_ascii=False))
    # 只保留最近 100 条
    r.ltrim(key, -100, -1)
    print(f"[写入] {key} <- {user}: {message}")


def get_history(session_id: str, limit: int = 20):
    """读取最近的 N 条对话"""
    key = f"chat:history:{session_id}"
    items = r.lrange(key, -limit, -1)
    print(f"\n[读取] 会话 {session_id} 最近 {len(items)} 条对话：")
    for raw in items:
        rec = json.loads(raw)
        print(f"   {rec['timestamp']} | {rec['user']}: {rec['message']}")
    return items


def clear_history(session_id: str):
    key = f"chat:history:{session_id}"
    r.delete(key)
    print(f"[清空] {key}")


if __name__ == "__main__":
    session_id = "session-001"

    # 先清空，方便重复测试
    clear_history(session_id)

    # 模拟一段对话
    add_message(session_id, "alice", "你好，我在测试 Redis")
    add_message(session_id, "bot",   "收到，Redis 已连接成功")
    add_message(session_id, "alice", "帮我记住这些历史")
    add_message(session_id, "bot",   "已经保存到 Redis List 中")

    # 读回历史
    get_history(session_id)

[清空] chat:history:session-001
[写入] chat:history:session-001 <- alice: 你好，我在测试 Redis
[写入] chat:history:session-001 <- bot: 收到，Redis 已连接成功
[写入] chat:history:session-001 <- alice: 帮我记住这些历史
[写入] chat:history:session-001 <- bot: 已经保存到 Redis List 中

[读取] 会话 session-001 最近 4 条对话：
   2026-11-12T10:53:30.495722 | alice: 你好，我在测试 Redis
   2026-11-12T10:53:30.496163 | bot: 收到，Redis 已连接成功
   2026-11-12T10:53:30.496394 | alice: 帮我记住这些历史
   2026-11-12T10:53:30.496603 | bot: 已经保存到 Redis List 中


/tmp/ipykernel_523/323117688.py:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


In [11]:




# kafka_producer.py
import json
from datetime import datetime

from kafka import KafkaProducer

# ---------- 配置 ----------
KAFKA_BOOTSTRAP = "127.0.0.1:9092"
KAFKA_TOPIC = "chat-messages"

# ---------- 初始化 ----------
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP,
    value_serializer=lambda v: json.dumps(v, ensure_ascii=False).encode("utf-8"),
    key_serializer=lambda k: k.encode("utf-8") if k else None,
    retries=3,
)


def send_message(session_id: str, user: str, message: str):
    payload = {
        "session_id": session_id,
        "user": user,
        "message": message,
        "timestamp": datetime.utcnow().isoformat(),
    }
    # 用 session_id 做 key，保证同会话落到同一分区（顺序一致）
    future = producer.send(KAFKA_TOPIC, key=session_id, value=payload)
    meta = future.get(timeout=10)
    print(f"[发送] topic={meta.topic} partition={meta.partition} "
          f"offset={meta.offset} -> {user}: {message}")


if __name__ == "__main__":
    session_id = "session-001"
    send_message(session_id, "alice", "第一条 Kafka 消息")
    send_message(session_id, "bot",   "第二条 Kafka 消息")
    send_message(session_id, "alice", "第三条 Kafka 消息")

    producer.flush()
    print("\n全部发送完成")

[发送] topic=chat-messages partition=0 offset=6 -> alice: 第一条 Kafka 消息
[发送] topic=chat-messages partition=0 offset=7 -> bot: 第二条 Kafka 消息
[发送] topic=chat-messages partition=0 offset=8 -> alice: 第三条 Kafka 消息

全部发送完成


/tmp/ipykernel_523/3020533898.py:25: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


In [ ]:
# kafka_consumer.py
import json

from kafka import KafkaConsumer

# ---------- 配置 ----------
KAFKA_BOOTSTRAP = "your-kafka-host:9092"
KAFKA_TOPIC = "chat-messages"
GROUP_ID = "chat-demo-group"

# ---------- 初始化 ----------
consumer = KafkaConsumer(
    KAFKA_TOPIC,
    bootstrap_servers=KAFKA_BOOTSTRAP,
    group_id=GROUP_ID,
    auto_offset_reset="earliest",   # 从头读，方便第一次体验
    enable_auto_commit=True,
    value_deserializer=lambda m: json.loads(m.decode("utf-8")),
)

if __name__ == "__main__":
    print(f"开始监听 topic={KAFKA_TOPIC} group={GROUP_ID} ...\n")
    try:
        for msg in consumer:
            payload = msg.value
            print(
                f"[接收] partition={msg.partition} offset={msg.offset} | "
                f"{payload['timestamp']} | {payload['user']}: {payload['message']}"
            )
    except KeyboardInterrupt:
        print("\n停止消费")
    finally:
        consumer.close()